In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
from pathlib import Path

from transformers import GenerationConfig

from pivotal_tokens.hf.loading import load_model, load_tokenizer
from pivotal_tokens.hf.sampling import batch_sample, extract_thinking_trace
from pivotal_tokens.hf.dataset import load_hotpotqa_dataset
from pivotal_tokens.extractor import SuccessProbabilityShiftExtractor
from pivotal_tokens.oracle import RegexOracle
from pivotal_tokens.repo import DictRepo
from pivotal_tokens.utils import setup_logging

/Users/maaxap/Workspace/Repos/phd/pivotal_tokens/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
HF_MODEL_ID = "Qwen/Qwen3-1.7B"
HF_DATASET_ID = "hotpotqa/hotpot_qa"
DICT_REPO_DIR = Path("repo")
ORACLE_ANSWER_REGEX = r"(?s)\s*(?:Answer:\s*)?(.*?)\s*(?=(?:<\|im_end\|>|<\|endoftext\|>|\Z))"

SYSTEM_PROMPT = ("Answer the following question directly, strictly without chain-of-thought after \"</think>\"."
                 "Keep the answer short (e.g., \"yes\" or \"no\" for binary questions, a person's full name if "
                 "the question expects a person, a country name if it asks about a country, etc.). Output the "
                 "answer strictly after the prefix \"Answer: \"  with no extra text.")

In [ ]:
setup_logging(logging.DEBUG)

model = load_model(model_id=HF_MODEL_ID, device="cpu")

In [ ]:
tokenizer = load_tokenizer(model_id=HF_MODEL_ID)


generation_config = GenerationConfig(temperature=0.6,
                                        top_p=0.95,
                                        top_k=20,
                                        min_p=0.1,
                                        max_new_tokens=4096,
                                        do_sample=True)


samples = load_hotpotqa_dataset(name="fullwiki", split="validation")
samples = samples[:30]


completions = batch_sample(samples=samples,
                           system_prompt=SYSTEM_PROMPT,
                           model=model,
                           tokenizer=tokenizer,
                           generation_config=generation_config,
                           batch_size=2)

traces = [extract_thinking_trace(c) for c in completions]

2025-12-08 22:41:20,505 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "HEAD /Qwen/Qwen3-1.7B/resolve/main/tokenizer_config.json HTTP/1.1" 307 0
2025-12-08 22:41:20,543 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen3-1.7B/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/tokenizer_config.json HTTP/1.1" 200 0
2025-12-08 22:41:20,681 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "GET /api/models/Qwen/Qwen3-1.7B/tree/main/additional_chat_templates?recursive=False&expand=False HTTP/1.1" 404 64
